<a href="https://www.kaggle.com/code/cartelsmith/how-to-cross-validation-lr-prediction?scriptVersionId=330398282" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "student-lifestyle-and-stress-dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sridevilavanyacse/student-lifestyle-and-stress-prediction-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import matplotlib.pyplot as plt

# Dataset General Info

In [ ]:
df.info()
print('--'*50)
df.head(10)

In [ ]:
# Missing Values
missing_pct = ((df.isna().sum()/25500)*100)
missing_pct.round(1).head(15)

sns.barplot(missing_pct.sort_values(ascending=False), palette='mako', orient='h')

plt.tight_layout()
plt.xlabel('% Missing')
plt.title('Percent Missing per Column')
plt.show();

In [ ]:
print(df.nunique())
df['Student_Type'] = df['Student_Type'].astype('category')


# Preliminary Data Overview (Light EDA)

In [ ]:
# Viewing distributions of numeric columns
num_cols = df.select_dtypes('number')


sns.pairplot(num_cols,diag_kind='kde',corner=True, dropna=True);


# Machine Learning: Model Construction

In [ ]:
# Split into training and validation sets
from sklearn.model_selection import train_test_split

features = df.drop(columns='Stress_Level')
target = df['Stress_Level']

Xtrain,xtest, Ytrain,ytest = train_test_split(features, target, random_state=12, train_size=.8, stratify=target)

In [ ]:
# Preprocess Data
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

num_cols = features.select_dtypes(include='number').columns.tolist()
cat_cols = features.select_dtypes(exclude='number').columns.tolist()


print(f'🔢The numeric columns are:\n{num_cols}')
print(f'\n\n🎏The categorical columns are:\n{cat_cols}')

# Building Transformers

num_transformer = make_pipeline(KNNImputer(n_neighbors=3))
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder())


# Building Preprocessor
preprocessor = ColumnTransformer([('num',num_transformer,num_cols),
                                  ('categories', categorical_transformer,cat_cols)
                                ])



In [ ]:
# Build Model
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model_pipeline = make_pipeline(preprocessor, model)



In [ ]:
# Evaluate Model
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate, cross_val_score
 
cv_score = cross_val_score(model_pipeline,Xtrain,Ytrain, cv=4, verbose=3)
print(f'\nThe cross validation score for each fold is: {cv_score}.\nAnd the avg score per fold is: {cv_score.mean():.2f}.')


# Machine Learning: Model Iteration and Improvement

In [ ]:
# Finding the ideal hyperparameters for this model using GridsearchCV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

#help(LogisticRegression) |Uncomment to see all params

param_sweep = [
    {
        'logisticregression__penalty': ['l2', None],
        'logisticregression__solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag'],
        
    },
    {
        'logisticregression__penalty': ['l1', 'l2'],
        'logisticregression__solver': ['liblinear'],
        
    },
    {
        'logisticregression__penalty': ['l1', 'l2', 'elasticnet', None],
        'logisticregression__solver': ['saga'],
        
    }
]

grid_search = RandomizedSearchCV(model_pipeline, param_distributions=param_sweep,
                                 n_iter = 25, cv=4, verbose=1)

grid_search.fit(Xtrain, Ytrain)

In [ ]:
# Evaluating the new optimized model
from sklearn.metrics import ConfusionMatrixDisplay

best_model = grid_search.best_estimator_

print(f'🥁🥁🥁...And the best parameters for this model are:\n {grid_search.best_params_}')
print(f"\n\nBest Cross-Validated Accuracy: {grid_search.best_score_:.2f} \n\n")

# Vizualizing performance using the BEST model
best_predictions = best_model.predict(xtest)

# Fixed Confusion Matrix plotting
disp = ConfusionMatrixDisplay.from_predictions(ytest, best_predictions, cmap='Blues')
plt.title("Optimized Model Confusion Matrix")
plt.show()